<a href="https://colab.research.google.com/github/siinwook/Deep-Learning-from-Scratch/blob/main/functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
def identify_functions(x):
  return x

In [ ]:
def step_functions(x):
  y=x>0
  return y.astype(int)

In [ ]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [ ]:
def relu(x):
  return np.maximum(0,x)

In [ ]:
def softmax(a):
  c = np.max(a,axis=1,keepdims=True) # save a.shape
  exp_a = np.exp(a-c)
  return exp_a / np.sum(exp_a,axis=1,keepdims=True)

In [ ]:
def sum_square_error(y,t): # y: f(x), t: target
  return np.sum((y-t)**2)

In [ ]:
def cross_entropy_error(y,t):
  if y.ndim==1:
    t=t.reshape(1,t.size)
    y=y.reshape(1,y.size)

  if t.size==y.size:  #change one-hot encoded target into numerical label
    t=t.argmax(axis=1)

  batch_size = y.shape[0]
  delta = 1e-7
  return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size

In [ ]:
def im2col(input_data, filter_h, filter_w, stride=1, pad=0): #(N,C,H,W) -> (N*outH*outW,C*FH*FW)
  N,C,H,W = input_data.shape

  out_h = (H + 2*pad - filter_h)//stride + 1
  out_w = (W + 2*pad - filter_w)//stride + 1

  img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')
  col = np.zeros((N,C,filter_h,filter_w,out_h,out_w)) #(N,C,FH,FW,outH,outW)

  for y in range(filter_h):
    y_max = y + stride*out_h
    for x in range(filter_w):
      x_max = x + stride*out_w
      col[:,:,y,x,:,:] = img[:,:,y:y_max:stride,x:x_max:stride]

  col = col.transpose(0,4,5,1,2,3).reshape(N*out_h*out_w,-1) #(N,C,FH,FW,outH,outW) -> (N,outH,outW,C,FH,FW) -> (N*outH*outW,C*FH*FW)
  return col #(N*outH*outW,C*FH*FW)

In [ ]:
def col2im(col, input_size, filter_h, filter_w, stride=1, pad=0): #(N*outH*outW,C*FH*FW) -> (N,C,H,W)
  N,C,H,W = input_size
  out_h = (H + 2*pad - filter_h)//stride + 1
  out_w = (W + 2*pad - filter_w)//stride + 1
  col = col.reshape(N,out_h,out_w,C,filter_h,filter_w).transpose(0,3,4,5,1,2) #(N*outH*outW,C*FH*FW) -> (N,outH,outW,C,FH,FW) -> (N,C,FH,FW,outH,outW)

  img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
  for y in range(filter_h):
    y_max = y + stride*out_h
    for x in range(filter_w):
      x_max = x + stride*out_w
      img[:,:,y:y_max:stride,x:x_max:stride] += col[:,:,y,x,:,:] #spread gradient (filter[y,x]) to each related pixel of img

  return img[:,:,pad:H+pad,pad:W+pad] #(N,C,H,W)